In [1]:
class FullBoxEnv:
    def reward(self, action):
        if action == "one-box":
            return 1000000       
        return 1001000           
 
class EmptyBoxEnv:
    def reward(self, action):
        if action == "one-box":
            return 0               
        return 1000

policies = ["one-box", "two-box"]
full = FullBoxEnv()
empty = EmptyBoxEnv()

In [ ]:
def ev_in_env(env, action):
    """
    Expected utility of an action in a single environment. deterministic, so 'expected' is just the reward.)
    """
    return env.reward(action)
 
 
def infra_value_same_bucket(envs, action):
    """
    Murphy picks the worst env for this action. Returns min over envs.
    """
    return min(ev_in_env(env, action) for env in envs)

def ev_with_policy_dependent_prior(action, omega_accuracy=0.99):
    """
    Omega is omega_accuracy% accurate. Your policy determines the prior.
    """
    if action == "one-box":
        p_full = omega_accuracy    
        p_empty = 1 - omega_accuracy
    else:
        p_full = 1 - omega_accuracy
        p_empty = omega_accuracy
 
    return (p_full * ev_in_env(full, action) +
            p_empty * ev_in_env(empty, action))


In [5]:
# Analysis 1: Same bucket (maximin)
print("Analysis 1: Same bucket — Murphy picks worst env")
print(f"\n{'Policy':<12} {'Full':<14} {'Empty':<14} {'Infra (min)':<14}")

best_name, best_val = None, -1
for action in policies:
    r_full = ev_in_env(full, action)
    r_empty = ev_in_env(empty, action)
    infra = infra_value_same_bucket([full, empty], action)
    print(f"{action:<12} ${r_full:>10,}   ${r_empty:>10,}   ${infra:>10,}")
    if infra > best_val:
        best_val = infra
        best_name = action
print(f"Optimal: {best_name} (${best_val:,})")

Analysis 1: Same bucket — Murphy picks worst env

Policy       Full           Empty          Infra (min)   
one-box      $ 1,000,000   $         0   $         0
two-box      $ 1,001,000   $     1,000   $     1,000
Optimal: two-box ($1,000)


In [16]:
# Analysis 2: Separate Buckets (Policy-dependent prior)
print("Analysis 2: Policy-dependent prior — Omega 99%")
print(f"\n{'Policy':<12} {'P(Full)':<10} {'P(Empty)':<10} {'EV':<20}")
best_name, best_val = None, -1
for action in policies:
    ev = ev_with_policy_dependent_prior(action, omega_accuracy=0.99)
    pf = 0.99 if action == "one-box" else 0.01
    pe = 1 - pf
    print(f"{action:<12} {pf:<10.2f} {pe:<10.2f} ${ev:>14,}")
    if ev > best_val:
        best_val = ev
        best_name = action
print(f"Optimal: {best_name} (${best_val:,})")

Analysis 2: Policy-dependent prior — Omega 99%

Policy       P(Full)    P(Empty)   EV                  
one-box      0.99       0.01       $     990,000.0
two-box      0.01       0.99       $11,000.00000000001
Optimal: one-box ($990,000.0)
